In [24]:
import json

# === Eingaben definieren ===
loesungs_ids = ["1_178", "14_210"]#, "9_210", "4_210"]

for loesung_id in loesungs_ids:
    instance_file = "Instanzen/Real_Life/2_piece/Construction_RealLife_2024_7_1_2.json"
    solution_file = "Solutions_RealLife/RealLife_2024_7_1_2/First_400_2/pareto_solutions_filtered.json"
    #output_file = f"Solutions_RealLife/RealLife_2024_7_1_2/First_400_2/generated_solution_{loesung_id}.json"
    output_file = f"generated_solution_{loesung_id}.json"


    # === Einlesen der Daten ===
    with open(instance_file, "r") as f:
        instance_data = json.load(f)

    with open(solution_file, "r") as f:
        solution_data = json.load(f)

    bp_dict = {bp["ID"]: bp for bp in instance_data["Bestellpositionen"]}
    maschinen_infos = {m["ID"]: m for m in instance_data["Maschinen"]}
    transportwege = instance_data.get("TransportwegeString", {})

    sol_entry = solution_data[loesung_id]

    # === Schritt 1: Arbeiter-Zuweisungen pro BP-ID aufbauen ===
    bp_to_workers = {}
    if "worker_route_plan" in sol_entry:
        for worker_id, bp_ids in sol_entry["worker_route_plan"].items():
            for bp_id in bp_ids:
                bp_to_workers.setdefault(bp_id, []).append(int(worker_id))

    # === Hilfsfunktion zur Anreicherung ===
    def enrich_bp(bp_id, maschinentyp="N/A", zugewiesene_arbeiter=None):
        bp = bp_dict.get(bp_id)
        if not bp:
            return None
        return {
            "ID": bp["ID"],
            "Start": bp["Start"],
            "Ende": bp["Ende"],
            "Dauer": bp["Dauer"],
            "Auftragsnummer": bp["Auftragsnummer"],
            "MaschinenTyp": maschinentyp,
            "AnbaugeraeteTypen": bp.get("AnbaugeraeteTypen", []),
            "ArbeiterQualifikationen": bp.get("ArbeiterQualifikationen", []),
            "ZugewieseneArbeiter": zugewiesene_arbeiter if zugewiesene_arbeiter else [],
            "zugewieseneMaschine": None,
            "Typ": "schicht"
        }

    # === Ergebnisstruktur initialisieren ===
    output_json = {
        "Version": "2025_01",
        "RechenzeitInSekunden": 0,
        "BerechnetAuftragBearbeitet": {},
        "MaschinenLoesung": {"Maschinenzuweisung": {}},
        "Arbeiterloesung": {"Arbeiterzuweisung": {}},
        "AnbaugeraeteLoesung": {"Anbaugeraetzuweisung": {}}
    }

    auftrag_status = {}

    # === Maschinenroute verarbeiten (inkl. Zuweisung von Arbeitern)
    def process_maschinenroute(routeplan, out_dict, prefix=""):
        for res_id, bp_ids in routeplan.items():
            key = f"{prefix}{res_id}"
            out_dict[key] = []
            for bp_id in bp_ids:
                zugewiesene_arbeiter = bp_to_workers.get(bp_id, [])
                enriched = enrich_bp(bp_id, zugewiesene_arbeiter=zugewiesene_arbeiter)
                if enriched:
                    out_dict[key].append(enriched)
                    auftrags_id = int(enriched["Auftragsnummer"])
                    auftrag_status[f"Auftrag {auftrags_id}"] = True

    # === Generische Verarbeitung für Arbeiter & Attachments
    def process_routeplan(routeplan, out_dict, prefix=""):
        for res_id, bp_ids in routeplan.items():
            key = f"{prefix}{res_id}"
            out_dict[key] = []
            for bp_id in bp_ids:
                enriched = enrich_bp(bp_id)
                if enriched:
                    out_dict[key].append(enriched)
                    auftrags_id = int(enriched["Auftragsnummer"])
                    auftrag_status[f"Auftrag {auftrags_id}"] = True

    # === Aufrufe
    if "machine_route_plan" in sol_entry:
        process_maschinenroute(sol_entry["machine_route_plan"], output_json["MaschinenLoesung"]["Maschinenzuweisung"], prefix="M_")

    if "worker_route_plan" in sol_entry:
        process_routeplan(sol_entry["worker_route_plan"], output_json["Arbeiterloesung"]["Arbeiterzuweisung"], prefix="Arbeiter_")

    if "attachment_route_plan" in sol_entry:
        process_routeplan(sol_entry["attachment_route_plan"], output_json["AnbaugeraeteLoesung"]["Anbaugeraetzuweisung"], prefix="ABG_")

    # === Noch nicht bearbeitete Aufträge auf False setzen
    alle_auftragsnummern = set(bp["Auftragsnummer"] for bp in instance_data["Bestellpositionen"])
    for auftragsnummer in alle_auftragsnummern:
        key = f"Auftrag {int(auftragsnummer)}"
        if key not in auftrag_status:
            auftrag_status[key] = False
    output_json["BerechnetAuftragBearbeitet"] = dict(sorted(auftrag_status.items(), key=lambda x: int(x[0].split()[1])))

    # === Erweiterung: Stammfahrer, Nutzung, Kilometer
    verletzungen_pro_maschine = {}
    maschine_genutzt = {}
    kilometer_pro_maschine = {}

    for maschine_id, einsatzliste in output_json["MaschinenLoesung"]["Maschinenzuweisung"].items():
        maschine_idx = int(maschine_id.split("_")[-1])
        maschine_data = maschinen_infos.get(maschine_idx, {})
        stammfahrer = set(maschine_data.get("StammfahrerStrings", []))  # strings!

        verletzungen = 0
        km_summe = 0.0
        last_auftragsnr = None

        for einsatz in einsatzliste:
            curr_auftragsnr = einsatz["Auftragsnummer"]
            zugewiesene_arbeiter = einsatz.get("ZugewieseneArbeiter", [])

            # === Stammfahrerprüfung ===
            has_stammfahrer = any(str(aid) in stammfahrer for aid in zugewiesene_arbeiter)
            if not has_stammfahrer:
                verletzungen += 1

            # === Transportdistanz (Auftragsnummern-basiert) ===
            if last_auftragsnr is not None:
                str_from = str(last_auftragsnr)
                str_to = str(curr_auftragsnr)
                km = transportwege.get(str_from, {}).get(str_to, 0.0)
                km_summe += km

            last_auftragsnr = curr_auftragsnr

        verletzungen_pro_maschine[maschine_id] = verletzungen
        maschine_genutzt[maschine_id] = len(einsatzliste) > 0
        kilometer_pro_maschine[maschine_id] = km_summe

    output_json["MaschinenLoesung"].update({
        "BerechneteStammfahrerVerletzungenProMaschine": verletzungen_pro_maschine,
        "BerechnetMaschineGenutzt": maschine_genutzt,
        "BerechneteKilometer": kilometer_pro_maschine,
        "BerechneteKilometerGesamt": sum(kilometer_pro_maschine.values()),
        "AnzahlGenutzterMaschinen": sum(maschine_genutzt.values()),
        "AnzahlStammfahrerVerletzungen": sum(verletzungen_pro_maschine.values())
    })

    # === Erweiterung: Arbeiter-Metriken ===

    arbeitswege = instance_data.get("ArbeitswegeString", {})

    arbeitsweg_pro_arbeiter = {}
    arbeiter_genutzt = {}

    for arbeiter_key, einsatzliste in output_json["Arbeiterloesung"]["Arbeiterzuweisung"].items():
        # arbeiter_key = "Arbeiter_5" → extrahiere numerische ID
        arbeiter_id = arbeiter_key.replace("Arbeiter_", "")
        
        km_summe = 0.0
        genutzt = len(einsatzliste) > 0

        for einsatz in einsatzliste:
            auftragsnr = einsatz["Auftragsnummer"]
            # Zugriff auf Arbeitswege von Arbeiter zu Auftrag
            km = arbeitswege.get(arbeiter_id, {}).get(str(auftragsnr), 0.0)
            km_summe += km

        arbeitsweg_pro_arbeiter[arbeiter_key] = 2*km_summe
        arbeiter_genutzt[arbeiter_key] = genutzt

    anzahl_genutzte_arbeiter = sum(arbeiter_genutzt.values())

    # === In JSON einfügen ===
    output_json["Arbeiterloesung"].update({
        "BerechneteKilometer": arbeitsweg_pro_arbeiter,
        "BerechneteKilometerGesamt": sum(arbeitsweg_pro_arbeiter.values()),
        "BerechnetArbeiterGenutzt": arbeiter_genutzt,
        "AnzahlGenutzterArbeiter": anzahl_genutzte_arbeiter
    })

    # === Erweiterung: Anbaugeräte-Metriken ===

    anbau_km = {}
    anbau_genutzt = {}

    for geraet_id, einsatzliste in output_json["AnbaugeraeteLoesung"]["Anbaugeraetzuweisung"].items():
        km_summe = 0.0
        genutzt = len(einsatzliste) > 0
        last_auftragsnr = None

        for einsatz in einsatzliste:
            curr_auftragsnr = einsatz["Auftragsnummer"]

            if last_auftragsnr is not None:
                str_from = str(last_auftragsnr)
                str_to = str(curr_auftragsnr)
                km = transportwege.get(str_from, {}).get(str_to, 0.0)
                km_summe += km

            last_auftragsnr = curr_auftragsnr

        anbau_km[geraet_id] = km_summe
        anbau_genutzt[geraet_id] = genutzt

    anzahl_genutzt = sum(anbau_genutzt.values())

    # === In JSON einfügen ===
    output_json["AnbaugeraeteLoesung"].update({
        "BerechneteKilometer": anbau_km,
        "BerechneteKilometerGesamt": sum(anbau_km.values()),
        "BerechnetAnbaugeraetGenutzt": anbau_genutzt,
        "AnzahlGenutzterAnbaugeraete": anzahl_genutzt
    })

    # === Export
    with open(output_file, "w") as f:
        json.dump(output_json, f, indent=4)

    print(f"✅ Datei geschrieben: {output_file}")

✅ Datei geschrieben: generated_solution_1_178.json
✅ Datei geschrieben: generated_solution_14_210.json
